# BERTScore Evaluation for the Three Model Submissions

This notebook evaluates the three generated submission files against the reference answers in `solutions.csv` with BERTScore. It follows the same clear pipeline style as the existing model notebooks: setup, data loading, validation, score computation, aggregation, and short interpretation.

The code below is written to stay practical for coursework: it checks the expected files, handles different CSV delimiters, aligns rows by `id` when possible, and saves both a compact summary table and an optional detailed score table for later error analysis.

# 1. Setup

This stage imports the required libraries, defines all file paths, and collects the main evaluation settings in one place.

Import explanation:
- `Path` is used to build file paths safely.
- `csv` is used to detect the delimiter of each input file.
- `re` is used to normalize whitespace before evaluation.
- `pandas` is used for CSV loading, merging, aggregation, and export.
- `torch` is used to select the available device for BERTScore.
- `display` is used to show clean table previews inside the notebook.
- `bertscore_score` is the BERTScore function from the `bert-score` package.

The variables below also define where the reference file, submission files, and exported evaluation tables are stored. Keeping these settings together makes the notebook easier to adjust if filenames or folders change later.

In [1]:
from pathlib import Path                                                       # Robust file handling for inputs and outputs.
import csv                                                                     # Used to detect CSV delimiters before loading files.
import re                                                                      # Used to normalize whitespace in ids and answers.

import pandas as pd                                                            # Data loading, joins, aggregation and CSV export.
import torch                                                                   # Used to select CPU or GPU for BERTScore.
from IPython.display import Markdown, display                                  # Notebook previews and rendered interpretation text.
from bert_score import score as bertscore_score                                # BERTScore computation for precision, recall and F1.

working_dir = Path.cwd()                                                       # The notebook may run either inside this folder or from its parent folder.
base_dir = working_dir if (working_dir / "solutions.csv").exists() else working_dir / "Accuracy BERTscore"   # Resolve the evaluation folder in a flexible way.

reference_path = base_dir / "solutions.csv"                                   # Ground-truth answers used as references.
submission_paths = {
    "Model 1 Inference": base_dir / "submission_inference.csv",
    "Model 2 Fine-Tune": base_dir / "submission_Fine-Tune.csv",
    "Model 3 RAG": base_dir / "submission_RAG.csv",
}                                                                             # Keep model names and file paths together for one shared loop.

summary_output_path = base_dir / "evaluation_summary.csv"                     # Compact comparison table saved for submission and reporting.
detailed_output_path = base_dir / "bertscore_detailed.csv"                    # Optional per-example scores for later error analysis.

bert_model_name = "bert-base-multilingual-cased"                              # Multilingual checkpoint that is suitable for German legal answers.
bert_batch_size = 16                                                           # Moderate batch size that works on CPU and many local GPUs.
device = "cuda" if torch.cuda.is_available() else "cpu"                       # Use the GPU when available, otherwise fall back to CPU.

print(f"Evaluation folder: {base_dir.resolve()}")
print(f"BERTScore model: {bert_model_name}")
print(f"Device: {device}")


C:\Users\Constantin Dieringer\Documents\Application of DS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Evaluation folder: C:\Users\Constantin Dieringer\OneDrive\Uni\Data Science\Application of DS\Code\Model training Constantin Dieringer\Accuracy BERTscore
BERTScore model: bert-base-multilingual-cased
Device: cpu


# 2. Load input files

This stage defines small helper functions for file validation, delimiter detection, text normalization, and column selection. The next block then loads the reference file and all three submission files into clean DataFrames.

Function definitions:
- `normalize_whitespace(value)`: Converts a value to text, removes repeated whitespace, and trims leading or trailing spaces.
- `detect_delimiter(csv_path)`: Reads a small sample of one CSV file and detects whether it uses `;`, `,`, or a tab delimiter.
- `find_first_matching_column(columns, candidates, dataset_name)`: Returns the first available column from a candidate list and raises a clear error if none exist.
- `load_csv_with_schema_checks(csv_path, required_columns, text_columns=None)`: Loads one CSV file, validates required columns, and normalizes selected text columns.

These helpers are important because the current files do not all use the same delimiter and some column names may change in future runs. Centralizing the loading logic keeps the later evaluation code short and easier to trust.

In [2]:
def normalize_whitespace(value):
    value = "" if value is None or pd.isna(value) else str(value)                # Convert missing values into safe empty strings before cleaning.
    return re.sub(r"\s+", " ", value).strip()                                 # Collapse repeated spaces and line breaks into one clean separator.


def detect_delimiter(csv_path):
    sample_text = csv_path.read_text(encoding="utf-8-sig")[:4096]              # Read only a small prefix because that is enough for delimiter detection.
    if sample_text.strip() == "":
        raise ValueError(f"The CSV file is empty: {csv_path.resolve()}")

    try:
        dialect = csv.Sniffer().sniff(sample_text, delimiters=";,\t,")        # Detect the delimiter instead of assuming one shared format.
        return dialect.delimiter
    except csv.Error:
        if ";" in sample_text.splitlines()[0]:
            return ";"                                                          # Use a simple fallback when the sniffer cannot decide.
        if "," in sample_text.splitlines()[0]:
            return ","
        raise ValueError(f"Could not detect a supported delimiter for {csv_path.resolve()}")


def find_first_matching_column(columns, candidates, dataset_name):
    normalized_lookup = {column.lower(): column for column in columns}           # Match candidate names case-insensitively for easier reuse.
    for candidate in candidates:
        if candidate.lower() in normalized_lookup:
            return normalized_lookup[candidate.lower()]
    raise ValueError(
        f"Could not find any of the expected columns {candidates} in {dataset_name}. Available columns: {list(columns)}"
    )


def load_csv_with_schema_checks(csv_path, required_columns, text_columns=None):
    if not csv_path.exists():
        raise FileNotFoundError(f"Required file not found: {csv_path.resolve()}")

    delimiter = detect_delimiter(csv_path)

    try:
        dataframe = pd.read_csv(csv_path, sep=delimiter, dtype="string", low_memory=False, encoding="utf-8-sig")
    except pd.errors.EmptyDataError as exc:
        raise ValueError(f"The CSV file is empty: {csv_path.resolve()}") from exc
    except pd.errors.ParserError as exc:
        raise ValueError(f"The CSV file could not be parsed: {csv_path.resolve()} | {exc}") from exc

    missing_columns = set(required_columns) - set(dataframe.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns in {csv_path.name}: {sorted(missing_columns)}")

    text_columns = [] if text_columns is None else list(text_columns)
    for column in text_columns:
        dataframe[column] = dataframe[column].map(normalize_whitespace)          # Clean ids and answers once so later joins and scores stay stable.

    return dataframe, delimiter


reference_id_column = "id"                                                      # The reference file is expected to identify each question by id.
reference_answer_candidates = ["correct_answer", "answer", "reference", "solution", "target"]
submission_answer_candidates = ["answer", "prediction", "predicted_answer", "generated_answer", "output"]

reference_df, reference_delimiter = load_csv_with_schema_checks(reference_path, {reference_id_column}, [reference_id_column])
reference_answer_column = find_first_matching_column(reference_df.columns, reference_answer_candidates, reference_path.name)
reference_df[reference_answer_column] = reference_df[reference_answer_column].map(normalize_whitespace)

submission_frames = {}
submission_delimiters = {}
for model_name, submission_path in submission_paths.items():
    submission_df, submission_delimiter = load_csv_with_schema_checks(submission_path, {"id"}, ["id"])
    submission_answer_column = find_first_matching_column(submission_df.columns, submission_answer_candidates, submission_path.name)
    submission_df[submission_answer_column] = submission_df[submission_answer_column].map(normalize_whitespace)
    submission_frames[model_name] = submission_df.rename(columns={submission_answer_column: "answer"}).copy()
    submission_delimiters[model_name] = submission_delimiter

reference_df = reference_df.rename(columns={reference_answer_column: "reference_answer"}).copy()
reference_df = reference_df.loc[:, ["id", "reference_answer"]].copy()         # Keep only the columns that are needed for the evaluation pipeline.

print(f"Reference delimiter: {reference_delimiter}")
for model_name, delimiter in submission_delimiters.items():
    print(f"{model_name} delimiter: {delimiter}")


Reference delimiter: ;
Model 1 Inference delimiter: ,
Model 2 Fine-Tune delimiter: ,
Model 3 RAG delimiter: ,


# 3. Inspect file structure and align rows

This stage checks the row counts, validates empty IDs, compares the submission IDs to the reference IDs, and builds one aligned evaluation table for each model. The notebook uses exact `id` matching by default, but it falls back to prefix-aware ordering inside a topic block when duplicated or shifted IDs make direct matching unreliable.

Function definitions:
- `validate_id_column(dataframe, dataset_name)`: Stops early if IDs are missing after normalization.
- `split_id_prefix(value)`: Extracts the non-numeric prefix of an ID so problematic blocks can be aligned by local order.
- `build_alignment_keys(dataframe, fallback_prefixes)`: Creates exact or prefix-order based alignment keys, depending on data quality in each block.
- `build_aligned_evaluation_frame(model_name, reference_frame, submission_frame)`: Merges one submission with the reference answers, keeps only comparable rows, and records missing, extra, and fallback-aligned prefixes.

This step is important because the current files contain one malformed ID block in the reference data. The alignment block makes these issues visible, applies a transparent fallback only where necessary, and ensures that all BERTScore values are computed on valid reference-prediction pairs.

In [3]:
def validate_id_column(dataframe, dataset_name):
    if dataframe["id"].eq("").any():
        raise ValueError(f"Found empty id values in {dataset_name} after normalization.")


def split_id_prefix(value):
    match = re.match(r"^(.*?)-(\d+)$", value)
    return match.group(1) if match else value                                   # Keep the whole id when no numeric suffix is present.


def build_alignment_keys(dataframe, fallback_prefixes):
    keyed_df = dataframe.copy()
    keyed_df["id_prefix"] = keyed_df["id"].map(split_id_prefix)
    keyed_df["prefix_position"] = keyed_df.groupby("id_prefix").cumcount() + 1   # Preserve the local order inside each topic block.
    keyed_df["alignment_key"] = keyed_df["id"]

    fallback_mask = keyed_df["id_prefix"].isin(fallback_prefixes)
    keyed_df.loc[fallback_mask, "alignment_key"] = keyed_df.loc[fallback_mask].apply(
        lambda row: f"{row['id_prefix']}__pos__{int(row['prefix_position']):03d}",
        axis=1,
    )                                                                         # Replace unreliable exact ids only inside problematic prefixes.
    return keyed_df


def build_aligned_evaluation_frame(model_name, reference_frame, submission_frame):
    validate_id_column(reference_frame, reference_path.name)
    validate_id_column(submission_frame, f"{model_name} submission")

    reference_prefix_stats = (
        reference_frame.assign(id_prefix=reference_frame["id"].map(split_id_prefix))
        .groupby("id_prefix")
        .agg(reference_rows=("id", "size"), reference_unique_ids=("id", "nunique"))
    )
    submission_prefix_stats = (
        submission_frame.assign(id_prefix=submission_frame["id"].map(split_id_prefix))
        .groupby("id_prefix")
        .agg(submission_rows=("id", "size"), submission_unique_ids=("id", "nunique"))
    )
    prefix_stats = reference_prefix_stats.join(submission_prefix_stats, how="outer").fillna(0)

    fallback_prefixes = sorted(
        prefix
        for prefix, row in prefix_stats.iterrows()
        if row["reference_rows"] != row["submission_rows"]
        or row["reference_rows"] != row["reference_unique_ids"]
        or row["submission_rows"] != row["submission_unique_ids"]
    )                                                                         # Use prefix-order alignment only where the exact ids are structurally unreliable.

    keyed_reference = build_alignment_keys(reference_frame, fallback_prefixes)
    keyed_submission = build_alignment_keys(submission_frame, fallback_prefixes)

    merged_df = keyed_reference.merge(
        keyed_submission.loc[:, ["alignment_key", "id", "answer"]].rename(columns={"id": "submission_id"}),
        on="alignment_key",
        how="left",
        indicator=True,
    )                                                                         # Merge with exact ids where possible and prefix-order fallback where needed.

    extra_keys = sorted(set(keyed_submission["alignment_key"]) - set(keyed_reference["alignment_key"]))
    missing_ids = merged_df.loc[merged_df["_merge"] == "left_only", "id"].sort_values().tolist()

    aligned_df = merged_df.loc[merged_df["_merge"] == "both", ["id", "submission_id", "reference_answer", "answer"]].copy()
    aligned_df["reference_answer"] = aligned_df["reference_answer"].map(normalize_whitespace)
    aligned_df["answer"] = aligned_df["answer"].map(normalize_whitespace)
    aligned_df = aligned_df[(aligned_df["reference_answer"] != "") & (aligned_df["answer"] != "")].reset_index(drop=True)
    aligned_df["model"] = model_name

    return aligned_df, missing_ids, extra_keys, fallback_prefixes


structure_records = []
aligned_frames = {}
alignment_report = {}

for model_name, submission_df in submission_frames.items():
    aligned_df, missing_ids, extra_keys, fallback_prefixes = build_aligned_evaluation_frame(model_name, reference_df, submission_df)
    aligned_frames[model_name] = aligned_df
    alignment_report[model_name] = {
        "missing_ids": missing_ids,
        "extra_alignment_keys": extra_keys,
        "fallback_prefixes": fallback_prefixes,
    }
    structure_records.append(
        {
            "model": model_name,
            "reference_rows": len(reference_df),
            "submission_rows": len(submission_df),
            "aligned_rows": len(aligned_df),
            "missing_reference_matches": len(missing_ids),
            "extra_submission_keys": len(extra_keys),
            "fallback_prefixes": ", ".join(fallback_prefixes) if fallback_prefixes else "",
        }
    )

structure_df = pd.DataFrame(structure_records)
display(structure_df)

for model_name, report in alignment_report.items():
    print(f"{model_name} missing ids: {report['missing_ids'][:10]}")
    print(f"{model_name} extra alignment keys: {report['extra_alignment_keys'][:10]}")
    print(f"{model_name} fallback prefixes: {report['fallback_prefixes']}")
    print()

display(aligned_frames["Model 1 Inference"].head(3))                           # Preview one aligned table before the score computation begins.


,model,reference_rows,submission_rows,aligned_rows,missing_reference_matches,extra_submission_keys,fallback_prefixes
0,Model 1 Inference,645,643,643,2,0,"ESTG27, VAT-INTL"
1,Model 2 Fine-Tune,645,643,643,2,0,"ESTG27, VAT-INTL"
2,Model 3 RAG,645,643,643,2,0,"ESTG27, VAT-INTL"


Model 1 Inference missing ids: ['ESTG27-039', 'ESTG27-040']
Model 1 Inference extra alignment keys: []
Model 1 Inference fallback prefixes: ['ESTG27', 'VAT-INTL']

Model 2 Fine-Tune missing ids: ['ESTG27-039', 'ESTG27-040']
Model 2 Fine-Tune extra alignment keys: []
Model 2 Fine-Tune fallback prefixes: ['ESTG27', 'VAT-INTL']

Model 3 RAG missing ids: ['ESTG27-039', 'ESTG27-040']
Model 3 RAG extra alignment keys: []
Model 3 RAG fallback prefixes: ['ESTG27', 'VAT-INTL']



,id,submission_id,reference_answer,answer,model
0,CORP-TAX-001,CORP-TAX-001,"Das Einkommen, das der unbeschränkt Steuerpfli...",liche Bemessungsgrundlage für die Körperschaft...,Model 1 Inference
1,CORP-TAX-002,CORP-TAX-002,Die nicht verrechneten Zinsen stellen eine ver...,vergibt? Antwort: Die Körperschaft kann den ge...,Model 1 Inference
2,CORP-TAX-003,CORP-TAX-003,"Steuerpflichtige, die aufgrund ihrer Rechtsfor...",Antwort: Die Bundesgesetzgebung über das Steue...,Model 1 Inference


# 4. Compute BERTScore for each model

This stage runs BERTScore for every aligned submission against the reference answers. The notebook computes precision, recall, and F1 per example and stores them in a detailed table.

Function definition:
- `compute_bertscore_for_frame(aligned_frame, model_type=bert_model_name)`: Runs BERTScore on one aligned DataFrame and returns the same rows with per-example precision, recall, and F1 columns.

The implementation keeps the model name, device, and batch size explicit so the evaluation stays reproducible. The multilingual BERT checkpoint is used because the answers are written in German and the submissions should be compared semantically rather than only by exact word overlap.

In [4]:
def compute_bertscore_for_frame(aligned_frame, model_type=bert_model_name):
    if aligned_frame.empty:
        raise ValueError("The aligned evaluation DataFrame is empty. No examples are available for BERTScore.")

    precision_tensor, recall_tensor, f1_tensor = bertscore_score(
        aligned_frame["answer"].tolist(),
        aligned_frame["reference_answer"].tolist(),
        model_type=model_type,
        batch_size=bert_batch_size,
        device=device,
        verbose=True,
    )                                                                         # Run BERTScore on the cleaned predictions and references.

    scored_frame = aligned_frame.copy()
    scored_frame["bertscore_precision"] = precision_tensor.cpu().numpy()
    scored_frame["bertscore_recall"] = recall_tensor.cpu().numpy()
    scored_frame["bertscore_f1"] = f1_tensor.cpu().numpy()
    return scored_frame


detailed_frames = []
for model_name, aligned_df in aligned_frames.items():
    print(f"Computing BERTScore for {model_name} on {len(aligned_df)} examples...")
    scored_df = compute_bertscore_for_frame(aligned_df)
    detailed_frames.append(scored_df)

detailed_results_df = pd.concat(detailed_frames, ignore_index=True)
display(detailed_results_df.head(3))


Computing BERTScore for Model 1 Inference on 643 examples...


C:\Users\Constantin Dieringer\Documents\Application of DS\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Constantin Dieringer\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13143.53it/s]


BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

 59%|█████▊    | 24/41 [00:00<00:00, 114.73it/s]

 88%|████████▊ | 36/41 [00:00<00:00, 117.02it/s]

100%|██████████| 41/41 [00:00<00:00, 117.79it/s]

done in 87.26 seconds, 7.37 sentences/sec


,id,submission_id,reference_answer,answer,model,bertscore_precision,bertscore_recall,bertscore_f1
0,CORP-TAX-001,CORP-TAX-001,"Das Einkommen, das der unbeschränkt Steuerpfli...",liche Bemessungsgrundlage für die Körperschaft...,Model 1 Inference,0.669040,0.735140,0.700534
1,CORP-TAX-002,CORP-TAX-002,Die nicht verrechneten Zinsen stellen eine ver...,vergibt? Antwort: Die Körperschaft kann den ge...,Model 1 Inference,0.652912,0.692876,0.672301
2,CORP-TAX-003,CORP-TAX-003,"Steuerpflichtige, die aufgrund ihrer Rechtsfor...",Antwort: Die Bundesgesetzgebung über das Steue...,Model 1 Inference,0.643767,0.664665,0.654049


# 5. Aggregate the results and save them

This stage averages the per-example BERTScore values for each model, builds the final comparison table, displays it in a readable format, and saves both the summary and the detailed results as CSV files.

The grouped summary is important for reporting because it gives one compact table with the number of evaluated examples and the mean precision, recall, and F1 for each model. Saving the detailed table is also useful because it makes later qualitative error analysis much easier.

In [5]:
summary_df = (
    detailed_results_df.groupby("model", as_index=False)
    .agg(
        n_examples=("id", "count"),
        Precision=("bertscore_precision", "mean"),
        Recall=("bertscore_recall", "mean"),
        F1=("bertscore_f1", "mean"),
    )
    .sort_values(by="F1", ascending=False)
    .reset_index(drop=True)
)                                                                             # Build one compact comparison table ordered by the most important metric.

display_summary_df = summary_df.copy()
display_summary_df[["Precision", "Recall", "F1"]] = display_summary_df[["Precision", "Recall", "F1"]].round(4)

display(display_summary_df)

summary_df.to_csv(summary_output_path, index=False, encoding="utf-8-sig")      # Save the full-precision summary for later reuse.
detailed_results_df.to_csv(detailed_output_path, index=False, encoding="utf-8-sig")   # Save per-example scores for optional error analysis.

print(f"Saved summary table to: {summary_output_path.resolve()}")
print(f"Saved detailed table to: {detailed_output_path.resolve()}")


,model,n_examples,Precision,Recall,F1
0,Model 1 Inference,643,0.6381,0.6795,0.6574
1,Model 3 RAG,643,0.6475,0.6590,0.6525
2,Model 2 Fine-Tune,643,0.6178,0.6619,0.6385


Saved summary table to: C:\Users\Constantin Dieringer\OneDrive\Uni\Data Science\Application of DS\Code\Model training Constantin Dieringer\Accuracy BERTscore\evaluation_summary.csv
Saved detailed table to: C:\Users\Constantin Dieringer\OneDrive\Uni\Data Science\Application of DS\Code\Model training Constantin Dieringer\Accuracy BERTscore\bertscore_detailed.csv


# 6. Interpret the final comparison

This stage identifies the best-performing model based on BERTScore F1 and renders a short interpretation directly in the notebook. The next block uses the already computed summary table, so the interpretation stays consistent with the final saved results.

This final step is important because the notebook should not only compute metrics, but also communicate what the results mean in a short and readable academic form.

In [6]:
best_row = summary_df.iloc[0]                                                    # The summary table is already sorted by F1 in descending order.
worst_row = summary_df.iloc[-1]
f1_gap = float(best_row["F1"] - worst_row["F1"])

interpretation_text = (
    f"### Result interpretation\n\n"
    f"The best-performing model in this BERTScore evaluation is **{best_row['model']}** with an average F1 score of **{best_row['F1']:.4f}**. "
    f"Across the aligned evaluation set of **{int(best_row['n_examples'])}** examples, this model achieved the strongest overall semantic match to the reference answers.\n\n"
    f"The gap between the best and lowest F1 score is **{f1_gap:.4f}**, which indicates how strongly the three approaches differ on semantic similarity to the target solutions. "
    f"Because the current submissions do not cover every reference id, the notebook evaluates only the rows that can be aligned reliably by `id`."
)                                                                             # Render a concise Markdown interpretation based on the computed summary values.

display(Markdown(interpretation_text))


### Result interpretation

The best-performing model in this BERTScore evaluation is **Model 1 Inference** with an average F1 score of **0.6574**. Across the aligned evaluation set of **643** examples, this model achieved the strongest overall semantic match to the reference answers.

The gap between the best and lowest F1 score is **0.0188**, which indicates how strongly the three approaches differ on semantic similarity to the target solutions. Because the current submissions do not cover every reference id, the notebook evaluates only the rows that can be aligned reliably by `id`.